# PlantDoctor - Lebanon orchard model (9 species, 35 classes)

Covers Lebanon's major fruit tree/orchard crops in one model: **Tomato, Apple, Cherry, Peach,
Grape** (all from the same PlantVillage dataset already used before) plus **Olive, Banana,
Citrus, Fig** (four additional real, verified Kaggle datasets - not PlantVillage, since it
doesn't cover these). Apricot was researched but has no clean labeled dataset available yet -
not included here, revisit later if one appears.

All 5 dataset sources were inspected directly via the Kaggle API before writing this notebook
(exact folder names below are verified, not guessed) - see
`docs/multi-species-expansion/PLAN.md` for the full research trail and source links.

1. **Settings (right sidebar) > Accelerator > GPU T4 x2** (or P100).
2. **Add Data** (right sidebar) and add ALL FIVE of these datasets:
   - `abdallahalidev/plantvillage-dataset` (Tomato/Apple/Cherry/Peach/Grape)
   - `habibulbasher01644/olive-leaf-image-dataset` (Olive)
   - `shifatearman/bananalsd` (Banana)
   - `dtrilsbeek/citrus-leaves-prepared` (Citrus - canker, black spot, greening, healthy)
   - `farziahossain/fig-leaf-original-dataset` (Fig)
3. Run all cells top to bottom - training happens here on Kaggle's GPU, then convert to TF.js
   **locally**, same as the tomato/multi-species notebooks (see this notebook's end).

## 1. Combine all 5 sources into one directory

Each dataset has its own folder layout (PlantVillage's `Species___Disease` siblings; Olive and
Citrus have their own train/test or train/validation splits; Banana ships an Original + an
already-augmented copy; Fig has no split at all). Rather than teaching the training pipeline
five different directory conventions, this materializes every class as a symlinked folder under
one common directory (`/kaggle/working/combined_data/<Species>___<Disease>/`), merging any
train/test/validation splits back together (we do our own split below, consistently, the same
way the tomato notebook already does). Symlinks (not copies) keep this fast and avoid using
extra disk.

In [ ]:
import os, pathlib

INPUT = pathlib.Path("/kaggle/input")
COMBINED_DIR = pathlib.Path("/kaggle/working/combined_data")
COMBINED_DIR.mkdir(exist_ok=True)

def find_dataset_root(slug):
    """Kaggle mounts an added dataset at /kaggle/input/<slug> by convention.
    Falls back to a prefix search in case Kaggle ever suffixes the folder name."""
    direct = INPUT / slug
    if direct.exists():
        return direct
    matches = [p for p in INPUT.iterdir() if p.is_dir() and p.name.startswith(slug)]
    assert matches, f"Could not find dataset '{slug}' under /kaggle/input - did you Add Data for it?"
    return matches[0]

def materialize(canonical_name, *source_dirs):
    """Symlink every image from one or more source folders into one canonical
    class folder, merging pre-existing train/test/validation splits back into
    a single pool (we apply our own consistent split later)."""
    dst = COMBINED_DIR / canonical_name
    dst.mkdir(parents=True, exist_ok=True)
    total = 0
    for i, src in enumerate(source_dirs):
        if not src.exists():
            print(f"  WARNING: missing source for {canonical_name}: {src}")
            continue
        for img in src.iterdir():
            if img.is_file():
                link = dst / f"s{i}_{img.name}"
                if not link.exists():
                    os.symlink(img, link)
                    total += 1
    print(f"{canonical_name}: {total} images")

In [ ]:
# --- PlantVillage: Tomato, Apple, Cherry, Peach, Grape ---
# Same fast-discovery trick as the earlier notebooks: PlantVillage's internal
# nesting (color/grayscale/segmented) is inconsistent across dataset
# versions, so find the "color" variant by walking rather than assuming a
# fixed path, and prune into any class folder rather than listing its images.
SPECIES = ["Tomato", "Apple", "Cherry", "Peach", "Grape"]

def matches_species(dirname):
    return "___" in dirname and any(dirname.startswith(s) for s in SPECIES)

candidate_dirs = set()
for root, dirnames, _files in os.walk(find_dataset_root("plantvillage-dataset")):
    keep = []
    for d in dirnames:
        if "___" in d:
            if matches_species(d):
                candidate_dirs.add(pathlib.Path(root))
            continue
        keep.append(d)
    dirnames[:] = keep

candidates = sorted(candidate_dirs)
assert candidates, "No matching PlantVillage class folders found"
color_dirs = [p for p in candidates if p.name == "color"]
PV_DIR = color_dirs[0] if color_dirs else candidates[0]
print("PlantVillage color dir:", PV_DIR)

pv_classes = sorted(d.name for d in PV_DIR.iterdir() if d.is_dir() and matches_species(d.name))
for cls in pv_classes:
    materialize(cls, PV_DIR / cls)

In [ ]:
# --- Olive: habibulbasher01644/olive-leaf-image-dataset ---
# Verified structure: dataset/{train,test}/{Healthy, aculus_olearius, olive_peacock_spot}
OLIVE = find_dataset_root("olive-leaf-image-dataset") / "dataset"
materialize("Olive___healthy", OLIVE / "train" / "Healthy", OLIVE / "test" / "Healthy")
materialize("Olive___bud_mite", OLIVE / "train" / "aculus_olearius", OLIVE / "test" / "aculus_olearius")
materialize("Olive___peacock_spot", OLIVE / "train" / "olive_peacock_spot", OLIVE / "test" / "olive_peacock_spot")

# --- Banana: shifatearman/bananalsd ---
# Verified structure: BananaLSD/{OriginalSet,AugmentedSet}/{healthy,cordana,pestalotiopsis,sigatoka}
# Use OriginalSet only - AugmentedSet is a pre-augmented copy, and we do our
# own augmentation in the model graph (see the tomato notebook's data_augmentation layer).
BANANA = find_dataset_root("bananalsd") / "BananaLSD" / "OriginalSet"
materialize("Banana___healthy", BANANA / "healthy")
materialize("Banana___cordana", BANANA / "cordana")
materialize("Banana___pestalotiopsis", BANANA / "pestalotiopsis")
materialize("Banana___sigatoka", BANANA / "sigatoka")

# --- Citrus: dtrilsbeek/citrus-leaves-prepared ---
# Verified structure: citrus_leaves_prepared/{train,validation}/{blackspot,canker,greening,healthy}
CITRUS = find_dataset_root("citrus-leaves-prepared") / "citrus_leaves_prepared"
materialize("Citrus___black_spot", CITRUS / "train" / "blackspot", CITRUS / "validation" / "blackspot")
materialize("Citrus___canker", CITRUS / "train" / "canker", CITRUS / "validation" / "canker")
materialize("Citrus___greening", CITRUS / "train" / "greening", CITRUS / "validation" / "greening")
materialize("Citrus___healthy", CITRUS / "train" / "healthy", CITRUS / "validation" / "healthy")

# --- Fig: farziahossain/fig-leaf-original-dataset ---
# Verified structure: "Fig Leaves Dataset"/{healthy,infected} - no train/test split, binary only
# (this dataset doesn't distinguish which fig disease - see remedies.json's fig_infected entry).
FIG = find_dataset_root("fig-leaf-original-dataset") / "Fig Leaves Dataset"
materialize("Fig___healthy", FIG / "healthy")
materialize("Fig___infected", FIG / "infected")

In [ ]:
classes = sorted(p.name for p in COMBINED_DIR.iterdir() if p.is_dir())
print(len(classes), "total classes:")
for c in classes:
    n = len(list((COMBINED_DIR / c).iterdir()))
    print(f"  {c}: {n} images")

Expect **35 classes**: 10 tomato + 4 apple + 2 cherry + 2 peach + 4 grape + 3 olive + 4 banana +
4 citrus + 2 fig. If any class shows 0 images, a dataset probably wasn't added via 'Add Data' -
check the WARNING lines printed above for which source path was missing.

## 2. Build train/val datasets

From here on, everything is identical to the tomato/multi-species notebooks - one combined
directory, one `image_dataset_from_directory` call, same normalization, same model, same export.
That reuse is deliberate: every fix already made (double-normalization, clean inference export,
Kaggle-side tensorflowjs conflicts) applies here unchanged.

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
DATA_DIR = COMBINED_DIR

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=classes)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=classes)

class_names = train_ds.class_names  # authoritative order used for the labels array
print(class_names)

# Normalize to [-1, 1] here in the data pipeline, matching exactly what
# App.js's classifyImage does (.div(127.5).sub(1)) before calling the model.
def normalize(x, y):
    return x / 127.5 - 1.0, y

train_ds = train_ds.map(normalize).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normalize).prefetch(tf.data.AUTOTUNE)
# No .cache(): caching decoded 224x224 images for 9 species in RAM risks the
# same out-of-memory kernel crash seen training on tomato alone.

## 3. Build the model (MobileNetV2 transfer learning)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False

pooling = tf.keras.layers.GlobalAveragePooling2D()
dropout = tf.keras.layers.Dropout(0.2)
classifier = tf.keras.layers.Dense(len(class_names), activation="softmax")

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = pooling(x)
x = dropout(x)
outputs = classifier(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train (frozen base)

35 classes and roughly 65-70k combined images (PlantVillage's ~50k+ five-species subset plus a
few thousand each from the other four sources) - noticeably more than the tomato-only run, so
expect a longer training time, still well within one Kaggle GPU session. Consider more epochs
than the tomato run (e.g. 10-12) given the larger, more varied class set.

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

## 5. Optional: fine-tune the top of MobileNetV2

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5)

## 6. Evaluate

Worth checking per-class accuracy here, not just the overall blended number - the four
non-PlantVillage sources have far fewer images per class than tomato does, and Fig's classes
are only binary (healthy/infected, no specific disease), so accuracy is likely uneven across
species rather than uniform.

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Overall validation accuracy: {acc:.3f}")

In [ ]:
import numpy as np

# Per-class accuracy on the validation set - flags any species/disease the
# blended overall accuracy above might be hiding.
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

for i, name in enumerate(class_names):
    mask = y_true == i
    n = mask.sum()
    if n == 0:
        print(f"  {name}: no validation samples")
        continue
    class_acc = (y_pred[mask] == i).mean()
    print(f"  {name}: {class_acc:.2f} ({n} val images)")

## 7. Save the trained model

Same reasoning as the earlier notebooks: converting to TF.js happens locally, not here.

In [ ]:
inference_inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input")
x = base_model(inference_inputs, training=False)
x = pooling(x)
x = dropout(x, training=False)
outputs = classifier(x)
inference_model = tf.keras.Model(inference_inputs, outputs)

inference_model.save("/kaggle/working/model.h5")

import json
metadata = {
    "modelName": "plantdoctor-lebanon-orchard-35class",
    "labels": class_names,
    "imageSize": IMG_SIZE,
}
with open("/kaggle/working/metadata.json", "w") as f:
    json.dump(metadata, f)

print("Saved:")
!ls -la /kaggle/working/model.h5 /kaggle/working/metadata.json

In [ ]:
!cd /kaggle/working && zip -q model_export.zip model.h5 metadata.json
print("Done - open the notebook's Output pane (after Save Version) and download model_export.zip")

## 8. Download

**Save Version > Save & Run All (Commit)**, wait for it to finish, then open that version's
**Output** tab and download `model_export.zip`.

## 9. Convert to TF.js locally

Identical steps to the tomato/multi-species models - reuse the same `model_export/` folder and
scripts (`export_weights.py`, `build_keras2.py`, `convert.py`) if you still have them, just point
them at this new `model.h5`. `build_keras2.py` reads `NUM_CLASSES` from `metadata.json`, so it
adapts automatically to the new 35-class output - no manual edit needed there.

See `CLAUDE.md`'s "Retraining the model" section for exactly why each conversion step is needed
(Keras 2 vs. 3 H5 format incompatibility, stale `tensorflow_decision_forests`/`tensorflow_hub`
imports inside `tensorflowjs`, deprecated `np.object` references).

## 10. Install into PlantDoctor

Copy `metadata.json` into the resulting `tfjs_model/` folder, then replace everything in
`public/model/` with those files. No app code changes needed - `App.js` and `remedies.js` are
both already generic over the label set (35 classes across 9 species, all already covered in
`assets/remedies.json`).